# Unit 4, Lecture 1: Many agents at once, fan-out and fan-in

Unit 3 ended with a graph that ran a **line**: classify, then enrich, then
assign. Real multi-agent systems are not lines. Several specialists look at the
same problem **at the same time**, and something gathers their answers back.

That shape is **fan-out** (one problem to many workers) and **fan-in** (many
answers into one). It is the first thing a plain pipeline genuinely cannot do.

The deep idea: **fan-in synchronises.** The combine step does not run until every
worker has finished. That waiting is coordination you would otherwise write by
hand, and get wrong. The graph does it for you.

Runs on the real `agent-framework` package, **offline**, because the workers are
plain functions. No lane needed.

## The three reviewers, each a plain function

In [ ]:
from cse476.orchestration import (
    security_review, priority_review, sentiment_review, REVIEWERS
)

# each reviewer is ordinary code, testable on its own, no framework needed
print("three independent reviewers:", [r.id for r in REVIEWERS])

Each reviewer answers one independent question about the ticket: is it a
security risk, is it urgent, is the customer angry. None depends on the others,
so there is no reason to run them in sequence.

## Fan-out then fan-in: the workflow

Read the two middle lines and you have the whole shape. `add_fan_out_edges`
sends the ticket to all three reviewers at once; `add_fan_in_edges` gathers their
findings into `combine`.

In [ ]:
from cse476.orchestration import build_review_workflow

wf = build_review_workflow()
print("workflow built:", type(wf).__name__)
print()
print("shape:  dispatch  =fan out=>  [security, priority, sentiment]  =fan in=>  combine")

## Run it, offline, deterministically

In [ ]:
from cse476.orchestration import run_review

# a nasty ticket lights up all three reviewers
print(await run_review("URGENT: someone tried to hack my account, this is terrible"))

# a calm ticket
print(await run_review("I have a small question about my invoice format"))

Three reviewers ran together, one verdict came back. Same input, same output,
every time, and not a token spent. Concurrency is usually the hardest thing to
test because of timing; here it is deterministic and offline, because the graph
does the coordination and the workers are pure.

## The key property: combine gets a LIST, called once

`combine` is **not** called three times, once per reviewer. It is called
**once**, with all three findings already gathered into a list. Look at the
verdict: it always has exactly three findings, never a partial set.

In [ ]:
verdict = await run_review("just a normal question")
findings = verdict.replace("Review complete: ", "").split(", ")
print("verdict:", verdict)
print("number of findings gathered:", len(findings))
print("combine saw all three at once, not one at a time")

## Proving the barrier: fan-in waits for everyone

This is the part worth seeing with your own eyes. Make one worker deliberately
slow, and confirm the combine step **still waits for it**. The join never runs
with a partial set.

In [ ]:
import asyncio
from agent_framework import WorkflowBuilder, WorkflowContext, executor

order = []

@executor(id="start")
async def start(x: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(x)

@executor(id="quick")
async def quick(x: str, ctx: WorkflowContext[str]) -> None:
    order.append("quick finished")
    await ctx.send_message("quick")

@executor(id="slow")
async def slow(x: str, ctx: WorkflowContext[str]) -> None:
    await asyncio.sleep(0.1)          # deliberately slow
    order.append("slow finished")
    await ctx.send_message("slow")

@executor(id="join")
async def join(results: list[str], ctx: WorkflowContext) -> None:
    order.append("JOIN ran")
    await ctx.yield_output(f"joined {len(results)} results")

barrier_wf = (WorkflowBuilder(start_executor=start)
    .add_fan_out_edges(start, [quick, slow])
    .add_fan_in_edges([quick, slow], join)
    .build())

result = await barrier_wf.run("go")
print("output:", result.get_outputs()[0])
print("order: ", order)
print()
print("Notice: JOIN ran LAST, only after the SLOW worker finished.")
print("You did not write that wait. The framework did.")

That "wait for everyone, then continue" is called a **barrier**, and writing
one by hand is a classic bug source: race conditions, partial results, a worker
that hangs. The graph gets it right, so you never write it and never debug it.
This is a concrete, interview-worthy reason to use the graph over hand-rolled
parallelism.

## The mapping, and when NOT to fan out

In [ ]:
from cse476.orchestration import ORCHESTRATION_MAP, sequential_vs_concurrent

for concept, tie in ORCHESTRATION_MAP.items():
    print(f"{concept:22} ->  {tie}")
print()
for k, v in sequential_vs_concurrent().items():
    print(f"{k:14}: {v}")

Concurrency is not free. If step B needs step A's result, they are **not**
independent and must stay a line (enrich needs the queue from classify). A
fan-in is only as fast as its **slowest** worker. And parallel systems are
harder to picture. Fan out genuinely independent work where speed matters; keep
everything else a line.

## Your turn

**1. Add a fourth reviewer.** For example a spam check or a language detector.
Add it to both the fan-out list and the fan-in list. Confirm the verdict now has
four findings, and that you changed only the edge lists.

**2. Prove the barrier yourself.** Put an `await asyncio.sleep` in one reviewer.
Confirm `combine` still waits for it and the verdict is never missing that
reviewer's finding.

**3. Line or fan?** List three tasks in your capstone. For each, decide honestly:
does it depend on another task, or is it independent? Draw the independent ones
as a fan. That discrimination is the design skill of this lecture.

In [ ]:
# your work here
